In [65]:
import csv
import json
import math
from datetime import datetime

In [142]:
def ler_transacoes(caminho:str):
    """
    Lê o CSV e retorna a lista de transações brutas
    """
    try:
        with open(caminho, "r", encoding="utf-8") as arquivo:
            dados = list(csv.DictReader(arquivo))

        return dados
    except FileNotFoundError:
        print("Erro! Arquivo não encontrado.")

def validar_formato_data_transacao(data:str):
    data_recebida = datetime.strptime(data, "%Y-%m-%d")

    return datetime.strftime(data_recebida, "%Y-%m")

def validar_valor_transacao(valor:str):
    valor_recebido = float(valor.replace(",", "."))

    if valor_recebido <= 0 or math.isnan(valor_recebido) or math.isinf(valor_recebido):
        raise ValueError("Transação inválida! Valor é menor ou igual a 0.")
    
    return valor_recebido

def validar_transacao(transacao:dict):
    """
    Valida uma única linha e retorna se for uma transação válida
    """
    try:
        id = int(transacao["id"])
        data_transacao = validar_formato_data_transacao(transacao["data"])
        valor = validar_valor_transacao(transacao["valor"])

        if transacao["cliente_id"].strip() == "":
            raise ValueError("Transação inválida! cliente_id está vazio.")

        if not transacao["tipo"].strip() in ["credito", "debito"]:
            raise ValueError("Transação inválida! Tipo é diferente de crédito/debito.")

        return {
                "id": id, 
                "data": data_transacao, 
                "cliente_id": transacao["cliente_id"],
                "tipo": transacao["tipo"],
                "valor": valor,
                "descricao": transacao["descricao"]
            }
    except ValueError:  
        return None

def exibir_resumo_validacao(transacoes:list):
    total_validacoes = len(transacoes)
    qtde_validas = 0
    qtde_invalidas = 0

    for transacao in transacoes:
        if not transacao:
            qtde_invalidas += 1
            continue

        qtde_validas += 1

    print(f"""Resumo da limpeza:\n
    - Total de linhas: {total_validacoes}
    - Linhas válidas: {qtde_validas}
    - Linhas inválidas: {qtde_invalidas}
    """)

def gerar_relatorio(transacoes:list):
    """
    Agrupa os dados, calcula as métricas e retorna um dicionário
    """
    relatorio = {}
    meses = sorted(set(transacao["data"] for transacao in transacoes))

    for mes in meses:
        relatorio[mes] = {
            "qtde_transacoes": 0,
            "credito": 0,
            "debito": 0,
            "saldo": 0,
            "media_transacoes": 0,
            "transacao_maior_valor": 0,
            "transacao_menor_valor": 0
        }

        for transacao in transacoes:
            if not transacao["data"] == mes: continue

            relatorio[mes]["qtde_transacoes"] += 1

            if transacao["tipo"] == "credito": relatorio[mes]["credito"] += transacao["valor"]
            if transacao["tipo"] == "debito": relatorio[mes]["debito"] += transacao["valor"]
            if transacao["valor"] > relatorio[mes]["transacao_maior_valor"]: relatorio[mes]["transacao_maior_valor"] = transacao["valor"]
            if relatorio[mes]["transacao_menor_valor"] == 0 or transacao["valor"] < relatorio[mes]["transacao_menor_valor"]: relatorio[mes]["transacao_menor_valor"] = transacao["valor"]

        relatorio[mes]["saldo"] = relatorio[mes]["credito"] - relatorio[mes]["debito"]
        relatorio[mes]["media_transacoes"] = (relatorio[mes]["credito"] + relatorio[mes]["debito"]) / relatorio[mes]["qtde_transacoes"]

    return relatorio

def identifica_transacao_suspeita(transacoes:list):
    limite_suspeito = 10000.00

    return [transacao for transacao in transacoes if transacao["valor"] > limite_suspeito]

def exibir_relatorio(relatorio:list, lista_suspeitos:list = None):
    """
    Formata e imprime os resultados no terminal
    """
    print("===== RELATÓRIO MENSAL =====")

    for mes in relatorio:
        print(f"""Mês: {mes}
        Transações: {relatorio[mes]["qtde_transacoes"]}
        Total crédito: R$ {relatorio[mes]["credito"]:.2f}
        Total débito:  R$ {relatorio[mes]["debito"]:.2f}
        Saldo:         R$ {relatorio[mes]["saldo"]:.2f}
        Média:         R$ {relatorio[mes]["media_transacoes"]:.2f}
        Maior valor:   R$ {relatorio[mes]["transacao_maior_valor"]:.2f}
        Menor valor:   R$ {relatorio[mes]["transacao_menor_valor"]:.2f}
        """.replace(".", ","))

    print("===== TRANSAÇÕES SUSPEITAS =====")
    if not lista_suspeitos: print("Não foi encontrada transação suspeita.")

    for transacao in lista_suspeitos:
        print(f"ID: {transacao["id"]} | Cliente: {transacao["cliente_id"]} | Data: {transacao["data"]} | Valor: R$ {transacao["valor"]}".replace(".", ","))

def salvar_json(relatorio:list):
    """
    Salva o resultado no arquivo relatorio.json
    """
    try:
        with open("relatorio.json", "w", encoding="utf-8") as arquivo:
            json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
    except FileNotFoundError:
        print("Não foi possível criar o arquivo json do relatório final.")

In [143]:
def main():
    transacoes_brutas = ler_transacoes("transacoes.csv")
    transacoes_processadas = [validar_transacao(transacao) for transacao in transacoes_brutas]
    
    exibir_resumo_validacao(transacoes_processadas)

    transacoes_validas = [transacao for transacao in transacoes_processadas if transacao]
    transacoes_suspeitas = identifica_transacao_suspeita(transacoes_validas)
    relatorio = gerar_relatorio(transacoes_validas)

    exibir_relatorio(relatorio, transacoes_suspeitas)

    salvar_json(relatorio)
    

In [145]:
main()

Resumo da limpeza:

    - Total de linhas: 120
    - Linhas válidas: 105
    - Linhas inválidas: 15
    
===== RELATÓRIO MENSAL =====
Mês: 2026-01
        Transações: 21
        Total crédito: R$ 20984,09
        Total débito:  R$ 9358,94
        Saldo:         R$ 11625,15
        Média:         R$ 1444,91
        Maior valor:   R$ 5835,15
        Menor valor:   R$ 142,20
        
Mês: 2026-02
        Transações: 21
        Total crédito: R$ 22835,32
        Total débito:  R$ 10868,25
        Saldo:         R$ 11967,07
        Média:         R$ 1604,93
        Maior valor:   R$ 5503,79
        Menor valor:   R$ 277,92
        
Mês: 2026-03
        Transações: 21
        Total crédito: R$ 11862,45
        Total débito:  R$ 9612,08
        Saldo:         R$ 2250,37
        Média:         R$ 1022,60
        Maior valor:   R$ 5716,26
        Menor valor:   R$ 89,46
        
Mês: 2026-04
        Transações: 21
        Total crédito: R$ 10096,08
        Total débito:  R$ 8734,15
        Sald